In [1]:
import pandas as pd
import numpy as np

# 1. Load the local CSV file directly
df = pd.read_csv('Global_Superstore2.csv', encoding='latin1')

# Clean up column names to strip out hidden spaces
df.columns = df.columns.str.strip()
print("Columns successfully loaded:", df.columns.tolist())

# 2. Convert Date formats safely
df['Order Date'] = pd.to_datetime(df['Order Date'])

# 3. RFM Calculation (Recency, Frequency, Monetary)
snapshot_date = df['Order Date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (snapshot_date - x.max()).days, # Recency
    'Order ID': 'nunique',                                 # Frequency
    'Sales': 'sum'                                         # Monetary
}).reset_index()

rfm.columns = ['Customer ID', 'Recency', 'Frequency', 'Monetary']

# 4. Create clean statistical scoring breaks (1 to 5 stars)
rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])

# 5. Assign Corporate Customer Segmentation Labels
def assign_segment(df_row):
    r = int(df_row['R_Score'])
    f = int(df_row['F_Score'])

    if r >= 4 and f >= 4:
        return 'Champions / Loyal Customers'
    elif r >= 3 and f >= 2:
        return 'Potential Loyalists'
    elif r >= 4 and f == 1:
        return 'New Customers'
    elif r <= 2 and f >= 3:
        return 'At Risk / Can\'t Lose Them'
    else:
        return 'Hibernating / Lost'

rfm['Customer_Segment'] = rfm.apply(assign_segment, axis=1)

# 6. Bring back Customer Name for easy dashboard identification
if 'Customer Name' in df.columns:
    cust_names = df[['Customer ID', 'Customer Name']].drop_duplicates(subset=['Customer ID'])
    rfm = pd.merge(rfm, cust_names, on='Customer ID', how='left')

# 7. Export the file right back into your local folder
rfm.to_csv('Superstore_RFM_Analytics.csv', index=False)
print("\n🔥 Success! Local file created as 'Superstore_RFM_Analytics.csv'")

Columns successfully loaded: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'City', 'State', 'Country', 'Postal Code', 'Market', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Cost', 'Order Priority']


/tmp/ipykernel_1190/919349961.py:12: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Order Date'] = pd.to_datetime(df['Order Date'])



🔥 Success! Local file created as 'Superstore_RFM_Analytics.csv'
